# ContractRisk — candidato OCR 19 (CPU)

La prueba visual 18 falló en utilidad (0/5 legibles). Este candidato **nuevo y separado** usa Tesseract 5, idiomas spa+eng y dos segmentaciones fijas (3 y 6). Se conservan los resultados Qwen: no se cambia su prompt ni se reemplazan cifras anteriores. La concordancia entre dos segmentaciones del mismo motor no es verificación independiente y su confianza no es probabilidad calibrada.

**No necesita GPU: seleccionar hardware Ninguno.** No necesita claves, Hugging Face ni API pagada. Una pasada sobre doce imágenes ya congeladas: ocho previas y cuatro sintéticas nuevas. La prueba evalúa lectura de píxeles, no respuestas QA, RAG, fuente contractual o calidad del despliegue. No cambia main ni el Space.

Ejecutar en orden; si falla, descargar evidencia desde la última celda. No repetir para buscar un acierto.

In [1]:
import os,sys,subprocess,tempfile,json
from pathlib import Path
SETUP_OK=False
ROOT=Path(tempfile.mkdtemp(prefix='contractrisk_ocr19_',dir='/content'))/'repo'
subprocess.run(['git','clone','--single-branch','--branch','Entrega/contractrisk-final','https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git',str(ROOT)],check=True)
os.chdir(ROOT)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Código:',COMMIT)
def run_visible(arguments,logfile):
    with open(logfile,'w',encoding='utf-8') as log:
        process=subprocess.Popen(arguments,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        status=process.wait()
    if status:raise RuntimeError('Proceso detenido; descargar diagnóstico con la última celda: '+logfile)


Código: c039a41bff4d0396c925179c38bf17c9ad7819a5


In [2]:
SETUP_OK=False
run_visible([sys.executable,'-m','pip','install','pillow==11.3.0'],'ocr19_install.log')
run_visible(['apt-get','update'],'ocr19_apt.log')
run_visible(['apt-get','install','-y','tesseract-ocr','tesseract-ocr-spa','tesseract-ocr-eng'],'ocr19_engine_install.log')
run_visible(['tesseract','--version'],'ocr19_engine_version.log')
run_visible(['tesseract','--list-langs'],'ocr19_languages.log')
run_visible([sys.executable,'-m','unittest','discover','-s','final/tests','-p','test_ocr19.py','-v'],'ocr19_tests.log')
from final.ocr_probe19 import load_protocol
print('Doce entradas congeladas:',[c['id'] for c in load_protocol()['cases']])
SETUP_OK=True


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [81.7 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,149 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,822 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Packages [61.7 kB]
Get:1

In [3]:
if not SETUP_OK:raise RuntimeError('Preparación incompleta. Descargar diagnóstico sin ejecutar OCR.')
RESULTS=ROOT/'ocr19_results'
run_visible([sys.executable,'-u','final/ocr_probe19.py','--output',str(RESULTS)],'ocr19_readings.log')


{"id": "full_original_once", "reading": {"output": {"legible": true, "texto_visible": "Identificador del contrato: CO1.PCCNTR.1088236\nEntidad: AEROCIVIL\nDepartamento: Distrito Capital de Bogotá\nCiudad: No Definido\nReferencia: 19001149 H4 2019\nDescripción del proceso: CONSTRUCCION DEL CERRAMIENTO DEL AEROPUERTO\nGOLFO DE MORROSQUILLO DE TOLU\nObjeto registrado: CONSTRUCCION DEL CERRAMIENTO DEL AEROPUERTO GOLFO DE\nMORROSQUILLO DE TOLU\nTipo de contrato: Obra\nModalidad de contratación: Selección Abreviada de Menor Cuantía\nEstado registrado en la captura: terminado\nFecha de firma registrada: 2024-03-27T11:30:19.253\nFecha de inicio registrada: 2019-09-10T12:00:00.000\nFecha de fin registrada: 2019-12-10T12:00:00.000\nDuración registrada: Dia(s)\nValor del contrato (COP): 250000000", "id_contrato": "CO1.PCCNTR.1088236", "objeto": "CONSTRUCCION DEL CERRAMIENTO DEL AEROPUERTO GOLFO DE MORROSQUILLO DE TOLU", "modalidad": "Selección Abreviada de Menor Cuantía", "valor_cop": "250000000"

## Interpretación y evidencia

Una lectura exacta no confirma autenticidad ni correspondencia con el corpus. En el caso de valor alterado, leer la cifra alterada es correcto como OCR pero NO como respuesta sobre el snapshot. `needs_source_and_answer_validation` significa que esas validaciones siguen pendientes. Un resultado con campo ausente debe bloquear preguntas sobre ese campo.

`completed_requires_review` no aprueba la integración. Guardamos las dos lecturas TSV, confianza, campos y rechazos. No hay búsqueda de parámetros ni reparación de dígitos a partir de referencias. Enviar ZIP y este notebook con salidas para revisión antes de integrar o desplegar.

In [4]:
import zipfile
from google.colab import files
metadata=ROOT/'ocr19_results/metadata.json'
print(metadata.read_text(encoding='utf-8') if metadata.exists() else 'No comenzó OCR o falta metadata. Revisar logs conservados.')
archive=ROOT.parent/'contractrisk_final_ocr19_resultados.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['ocr19_results','final/evidence/ocr19_protocol','final/evidence/visual18_protocol']:
        base=ROOT/folder
        if base.exists():
            for item in sorted(base.rglob('*')):
                if item.is_file():z.write(item,item.relative_to(ROOT))
    for item in sorted(ROOT.glob('ocr19_*.log')):z.write(item,item.name)
    z.writestr('checkout_commit.txt',COMMIT+'\n')
    z.writestr('setup_state.json',json.dumps({'setup_ok':SETUP_OK}))
files.download(str(archive))


{
  "status": "completed_requires_review",
  "purpose": "post_hoc_ocr_only_pilot_not_integrated_scorecard",
  "api_calls": 0,
  "gpu_used": false,
  "model_or_prompt_search": false,
  "started_utc": "2026-11-20T00:50:15.574223+00:00",
  "settings": {
    "version": "ocr-candidate-19",
    "language": "spa+eng",
    "oem": 1,
    "psm": [
      3,
      6
    ],
    "min_word_confidence": 80.0,
    "timeout_per_pass_seconds": 30,
    "max_pixels": 12000000,
    "retries": 0,
    "note": "Two layouts of ONE engine; not independent readers or calibrated confidence."
  },
  "source_commit": "c039a41bff4d0396c925179c38bf17c9ad7819a5",
  "protocol_sha256": "de78945ea0d314a0a1049cd9711b3ec50ff502abd9ac0351d8e4460969326f08",
  "tesseract_version": "tesseract 5.3.4",
  "language_data_sha256": {
    "spa": "6f2e04d02774a18f01bed44b1111f2cd7f3ba7ac9dc4373cd3f898a40ea6b464",
    "eng": "7d4322bd2a7749724879683fc3912cb542f19906c83bcc1a52132556427170b2"
  },
  "languages": "List of available languag

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>